## Loading data

In [1]:
# Importing the necessary libraries
import pandas as pd
import numpy as np
import kennard_stone as ks
pd.options.plotting.backend = 'plotly'  # setting plotly as the backend for pandas plotting

# Add parent directory to sys.path so local module 'synthetic' (one level up) can be imported
import sys
from pathlib import Path # for path manipulations
parent_dir = Path.cwd().parent.parent.resolve() # move two levels up from current working directory
if str(parent_dir) not in sys.path: # check to avoid duplicates
    sys.path.insert(0, str(parent_dir)) # insert at the start of sys.path to prioritize local modules

# Loading a soil spectral dataset based on X-ray fluorescence (XRF)
data_complete = pd.read_csv(f'{parent_dir}/XRF_databases/soil/plsda/soil.csv', sep=';') # local copy of Toledo 2022 dataset (os ... indica para omitir o caminho longo)
data = data_complete.loc[:, '1':'15']

In [2]:
# Split dataset by class and create calibration/prediction sets using Kennard-Stone (as in original pipeline)
data_A = data_complete[data_complete['Class'] == 'A'].reset_index(drop=True)
data_B = data_complete[data_complete['Class'] == 'B'].reset_index(drop=True)

# splitting the data into calibration and prediction sets by kennard-stone algorithm
XA_cal, XA_pred = ks.train_test_split(data_A.loc[:, '1':'15'], test_size=0.30)  # class A
XA_cal = XA_cal.reset_index(drop=True)
XA_pred = XA_pred.reset_index(drop=True)

XB_cal, XB_pred = ks.train_test_split(data_B.loc[:, '1':'15'], test_size=0.30)  # class B
XB_cal = XB_cal.reset_index(drop=True)
XB_pred = XB_pred.reset_index(drop=True)

Xcalclass = pd.concat([XA_cal, XB_cal], axis=0).reset_index(drop=True)  # concatenating both classes
Xpredclass = pd.concat([XA_pred, XB_pred], axis=0).reset_index(drop=True)
ycalclass = pd.Series(['A']*XA_cal.shape[0] + ['B']*XB_cal.shape[0])  # target for calibration set
ypredclass = pd.Series(['A']*XA_pred.shape[0] + ['B']*XB_pred.shape[0])  # target for prediction set

# preprocessings
import preprocessings as prepr  # preprocessing methods for XRF data

Xcalclass_prep, mean_calclass, mean_calclass_poisson  = prepr.poisson(Xcalclass, mc=True)
Xpredclass_prep = ((Xpredclass/np.sqrt(mean_calclass)) - mean_calclass_poisson)

2026-01-24 17:14:10,696 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-24 17:14:10,700 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-24 17:14:10,777 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.

2026-01-24 17:14:10,795 - kennard_stone.utils._pairwise:114[INFO] - Calculating pairwise distances using scikit-learn.



In [3]:
# PLS-DA with optimized latent variables
from modeling import pls_optimized

plsda_results = pls_optimized(
    Xcalclass_prep, 
    ycalclass,
    LVmax=4,
    Xpred=Xpredclass_prep,
    ypred=ypredclass,
    aim='classification',
    cv=10
)

# Convenience references used later
pls_model = plsda_results[3]               # fitted PLS model
vip_scores_mat = plsda_results[4]          # VIP scores matrix (features × LV)
y_pred_cont = plsda_results[5].iloc[:, -1] # continuous predictions for Xcalclass (used for MI/Cov)

# plotando o vip scores rapidamente
vip_scores_mat.T.plot()

In [4]:
# calculando a covariancia entre cada variável espectral e a predição do modelo PLS-DA
cov_scores = []
y_pred = plsda_results[5].iloc[:,-1].values # using the continuous predictions from LV=3
for col in Xcalclass_prep.columns:
    x_values = Xcalclass_prep[col].values
    covariance = np.cov(x_values, y_pred)[0, 1] # covariance between x and y
    cov_scores.append(covariance)
cov_scores_df = pd.DataFrame(cov_scores, index=Xcalclass_prep.columns, columns=['Covariance'])
cov_scores_df = np.abs(cov_scores_df)
cov_scores_df.plot()

In [5]:
# establishing spectral cuts based on expert knowledge of XRF spectra
spectral_cuts = [
('background1', 1.0, 1.33),
('Al', 1.33, 1.63),
('Si', 1.63, 1.86),
('P', 1.86, 2.10),
('background2', 2.10, 2.19),
('S', 2.19, 2.44),
('background3', 2.44, 2.55),
('Rh L + Ar', 2.55, 3.10),
('background4', 3.10, 3.21),
('K', 3.21, 3.42),
('background5', 3.42, 3.53),
('Ca ka', 3.53, 3.84),
('Ca kb', 3.84, 4.14),
('background6', 4.14, 4.37),
('Ti ka', 4.37, 4.66),
('background7', 4.66, 4.75),
('Ti kb', 4.75, 5.12),
('Cr', 5.12, 5.77),
('Mn', 5.77, 6.02),
('background8', 6.02, 6.13),
('Fe ka', 6.13, 6.68),
('background9', 6.68, 6.80),
('Fe kb', 6.80, 7.30),
('background10', 7.30, 7.91),
('Cu', 7.91, 8.20),
('background11', 8.20, 10.69),
('Fe ka + Ti ka', 10.69, 11.14),
('background12', 11.14, 12.55),
('sum Fe' , 12.55, 13.1),
('background13', 13.1, 15.0)
]

import explaining as exp
spectral_zones_class = exp.extract_spectral_zones(Xcalclass_prep, spectral_cuts)
zone_sums_df = exp.aggregate_spectral_zones(spectral_zones_class, aggregator='extreme')
predicates_quantiles = exp.predicates_by_quantiles(zone_sums_df, [0.2, 0.4, 0.6, 0.8])
co_occurrence_matrix_df = predicates_quantiles[2]
predicate_info_dict = exp.create_predicate_info_dict(
    predicates_df=predicates_quantiles[0],
    predicate_indicator_df=predicates_quantiles[1],
    zone_aggregated_df=zone_sums_df,
    y_predicted_numeric=y_pred_cont
)

## VIP, Regression Coefficients e SHAP (como no original)

In [6]:
# VIP scores por energia
vip_scores_df = pd.DataFrame({
    'energy': vip_scores_mat.T.index,
    'VIP_Score': vip_scores_mat.T.iloc[:,0].values
})
vip_scores_df = vip_scores_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)
energy_to_zone_vip = {}
for zone_name, start, end in spectral_cuts:
    for e in vip_scores_df['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_vip[e] = zone_name
vip_scores_df['Zone'] = vip_scores_df['energy'].map(energy_to_zone_vip)
vip_scores_unique_df = vip_scores_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
vip_scores_unique_df = vip_scores_unique_df.sort_values(by='VIP_Score', ascending=False).reset_index(drop=True)

# Coeficientes de regressão do PLS
reg_vet = pd.DataFrame(pls_model.coef_, columns=pls_model.feature_names_in_).T
reg_vet.insert(0, 'energy', reg_vet.index)
reg_vet = reg_vet.reset_index(drop=True)
reg_vet.columns = ['energy','Reg_coef']
reg_vet['Abs_Reg_coef'] = reg_vet['Reg_coef'].abs()
reg_vet = reg_vet.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)
energy_to_zone_reg = {}
for zone_name, start, end in spectral_cuts:
    for e in reg_vet['energy']:
        ef = float(e)
        if start <= ef <= end:
            energy_to_zone_reg[e] = zone_name
reg_vet['Zone'] = reg_vet['energy'].map(energy_to_zone_reg)
reg_vet_unique_df = reg_vet.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
reg_vet_unique_df = reg_vet_unique_df.sort_values(by='Abs_Reg_coef', ascending=False).reset_index(drop=True)

# **bagging - covariance**

In [7]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_cov = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist
    # Calcular MI
    cov_results_dict_seed = exp.calculate_predicate_metrics(
        bags_result=bags_result_seed,
        metric='covariance', # covariance ou mutual_information
        threshold=0.01, # threshold para cortar predicados irrelevantes
        n_neighbors=5
    )
    # Salvar no dicionário principal
    all_results_cov[seed] = {
        'bags_result': bags_result_seed,
        'cov_results_dict': cov_results_dict_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_cov[seed]['bags_result'],
        mi_results_dict=all_results_cov[seed]['cov_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_cov_by_seed = {}
for seed in random_seeds:
    DG = graphs_by_seed[seed]
    lrc_cov_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_cov_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_cov_by_seed[seed] = lrc_cov_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_cov_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_cov_df_seed = lrc_cov_by_seed[seed].rename(columns={'Node': f'Predicate_Cov_Seed_{seed}'})
    lrc_cov_all_seeds_df = pd.concat([lrc_cov_all_seeds_df, lrc_cov_df_seed[[f'Predicate_Cov_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_cov_unique_by_seed = {}
for seed, lrc_df in lrc_cov_by_seed.items():
    lrc_cov_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_cov_unique_df = lrc_cov_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_cov_unique_by_seed[seed] = lrc_cov_unique_df

lrc_cov_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58


Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 180 | Descartados: 60
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Calculando Covariance para Predicados
Métrica: covariance
Threshold: 0.01

Processando semente: 1

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 184 | Descartados: 56
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 186 | Descartados: 54
Bag_4 | A

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...

Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


,Predicate_Cov_Seed_0,Predicate_Cov_Seed_1,Predicate_Cov_Seed_2,Predicate_Cov_Seed_3
0,Ca ka > -0.22,Ca ka > -0.72,Ca ka > -0.22,Ca ka > -0.22
1,Ca ka > -0.72,Fe ka > -1.50,Ca ka > -0.72,Ca ka > -0.72
2,Ca ka > -0.47,Ca ka > -0.47,Ca ka > -0.47,Ca ka > -0.47
3,Fe ka > -1.50,Ca ka > -0.22,Fe ka > -1.50,Fe ka > -1.50
4,Ca kb <= 0.20,Fe ka > -0.44,Ca ka <= 0.58,Fe ka <= 1.45
...,...,...,...,...
80,NaN,background10 <= 0.15,Class_A,background7 > 0.09
81,NaN,P > 0.12,Class_B,Al > 0.16
82,NaN,Class_A,NaN,background9 <= 0.08
83,NaN,Class_B,NaN,Class_A


In [8]:
from collections import defaultdict

# Coletar posições de cada predicado em cada seed do lrc_pert_all_seeds_df
positions_dict_lrc = defaultdict(list)

# Iterar sobre cada coluna do dataframe lrc_pert_all_seeds_df
for col in lrc_cov_all_seeds_df.columns:
    # Pegar os predicados da coluna (não-nulos)
    predicates_in_seed = lrc_cov_all_seeds_df[col].dropna().tolist()
    
    # Para cada predicado, guardar sua posição (1-based)
    for position, predicate in enumerate(predicates_in_seed, start=1):
        positions_dict_lrc[predicate].append(position)

# Calcular média e número de aparições
results_lrc = []
for predicate, positions in positions_dict_lrc.items():
    zone_row = predicates_quantiles[0].loc[predicates_quantiles[0]['rule'] == predicate, 'zone']
    zone_value = zone_row.values[0] if not zone_row.empty else None
    results_lrc.append({
        'Predicate': predicate,
        'Mean_Position': np.mean(positions),
        'Appearances': len(positions),
        'Zone': zone_value
    })

# Ordenar: menor posição média primeiro, mais aparições em caso de empate
ranking_lrc_cov_df = pd.DataFrame(results_lrc).sort_values(
    by=['Mean_Position', 'Appearances'], 
    ascending=[True, False]
).reset_index(drop=True)

# Lista final ordenada
lista_ordenada_lrc = ranking_lrc_cov_df['Predicate'].tolist()
ranking_lrc_cov_df

,Predicate,Mean_Position,Appearances,Zone
0,Ca ka > -0.22,1.75,4,Ca ka
1,Ca ka > -0.72,1.75,4,Ca ka
2,Ca ka > -0.47,3.00,4,Ca ka
3,Fe ka > -1.50,3.50,4,Fe ka
4,Ca ka <= 0.58,9.50,4,Ca ka
...,...,...,...,...
87,background9 <= 0.08,81.00,2,background9
88,P > 0.12,81.00,2,P
89,background10 <= 0.15,81.00,1,background10
90,Class_A,81.25,4,None


In [9]:
ranking_cov_lrc_unique_df = ranking_lrc_cov_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
ranking_cov_lrc_unique_df['Zone']

0             Ca ka
1             Fe ka
2             Ca kb
3                Al
4                 K
5                Si
6                Mn
7             Ti ka
8             Ti kb
9             Fe kb
10                P
11     background13
12           sum Fe
13               Cr
14               Cu
15      background9
16     background12
17                S
18     background11
19      background6
20      background1
21    Fe ka + Ti ka
22      background7
23        Rh L + Ar
24      background8
25     background10
26             None
Name: Zone, dtype: object

# **Perturbation**

In [10]:
import explaining as exp

# LISTA DE SEMENTES A TESTAR
random_seeds = [0, 1, 2, 3]

all_results_pert = {}
training_samples = len(Xcalclass)

# LOOP: PROCESSAR CADA SEMENTE
y_predicted_numeric = plsda_results[5].iloc[:, -1] # predições numéricas do modelo

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando semente: {seed}")
    print(f"{'='*70}\n")
    # Bagging
    bags_result_seed = exp.bagging_predicates(
        zone_sums_df=zone_sums_df,
        y_predicted_numeric=y_predicted_numeric,
        predicates_df=predicates_quantiles[0],
        n_bags=10,
        #n_predicates_per_bag=40,
        n_samples_per_bag=int(training_samples*0.8), # 80 % da base para amostrar (convertido para int)
        min_samples_per_predicate=int(training_samples*0.2), # 20 % da base para limitar (convertido para int)
        replace=False,
        sample_bagging=True,
        predicate_bagging=False,
        random_seed=seed
    )
    # Inserir classe prevista
    for bag_name, pred_dict in bags_result_seed.items(): # iterando sobre cada bag
        for pred_rule, df_info in pred_dict.items():
            df_info['Class_Predicted'] = np.where(df_info['Predicted_Y'] >= 0.5, 'A', 'B') # binarizando com threshold 0.5, A = eut, B = dist

    pert_results_seed = exp.calculate_predicate_perturbation(
        estimator=pls_model,
        Xcalclass_prep=Xcalclass_prep,
        folds_struct=bags_result_seed,
        predicates_df=predicates_quantiles[0],
        spectral_cuts=spectral_cuts,
        perturbation_value=0,
        metric='mean_relative_dev',   # Média com sinal (pode ser negativo)
        verbose=True
    )

    # Remove todos os valores iguais a zero de todos os bags em perm_results[bag]["Permutation"] e salva como perm_results_thresholded
    # pert_results_seed_thresholded = {}
    # for bag, df in perm_results_seed.items():
    #     # Verifica se é um DataFrame e se a coluna 'Permutation' existe
    #     if isinstance(df, pd.DataFrame) and 'Permutation' in df.columns:
    #         filtered_df = df[df['Permutation'] > 0].copy()
    #         pert_results_seed_thresholded[bag] = filtered_df
    #     else:
    #         # Se não for DataFrame esperado, apenas copia
    #         pert_results_seed_thresholded[bag] = df

    # Salvar no dicionário principal
    all_results_pert[seed] = {
        'bags_result': bags_result_seed,
        'pert_results_dict': pert_results_seed
    }

# CONSTRUÇÃO DE GRAFOS PARA MÚLTIPLAS SEMENTES (LOOP EXTERNO)
# Dicionário para armazenar grafos
graphs_pert_by_seed = {}

for seed in random_seeds:
    print(f"\n{'='*70}")
    print(f"Processando Grafo - Semente: {seed}")
    print(f"{'='*70}\n")
    # Construir grafo para esta semente
    DG = exp.build_predicate_graph(
        bags_result=all_results_pert[seed]['bags_result'],
        mi_results_dict=all_results_pert[seed]['pert_results_dict'],
        co_occurrence_matrix_df=co_occurrence_matrix_df,
        predicates_df=predicates_quantiles[0],
        random_state=seed,
        show_details=True
    )
    # Armazenar grafo
    graphs_pert_by_seed[seed] = DG  

# Calcular LRC usando a função pronta do explaining.py
lrc_pert_by_seed = {}
for seed in random_seeds:
    DG = graphs_pert_by_seed[seed]
    lrc_pert_df_seed = exp.calculate_lrc_single_graph(DG, predicates_quantiles[0])
    lrc_pert_df_seed['Seed'] = seed  # Adicionar coluna com a semente
    lrc_pert_by_seed[seed] = lrc_pert_df_seed

# junando todas as colunas 'Node' de lrc_by_seed em um único dataframe
lrc_pert_all_seeds_df = pd.DataFrame()
for seed in random_seeds:
    lrc_pert_df_seed = lrc_pert_by_seed[seed].rename(columns={'Node': f'Predicate_pert_Seed_{seed}'})
    lrc_pert_all_seeds_df = pd.concat([lrc_pert_all_seeds_df, lrc_pert_df_seed[[f'Predicate_pert_Seed_{seed}']]], axis=1)

# vamos filtrar lrc_by_seed em cada semente para manter apenas as zonas espectrais únicas com maior LRC em um mesmo dataframe
lrc_pert_unique_by_seed = {}
for seed, lrc_df in lrc_pert_by_seed.items():
    lrc_pert_unique_df = lrc_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
    lrc_pert_unique_df = lrc_pert_unique_df.sort_values(by='Local_Reaching_Centrality', ascending=False).reset_index(drop=True)
    lrc_pert_unique_by_seed[seed] = lrc_pert_unique_df

lrc_pert_all_seeds_df # exibindo o dataframe consolidado com predicados de todas as sementes


Processando semente: 0

Bag_1 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_2 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 182 | Descartados: 58
Bag_3 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_4 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_5 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_6 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_7 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 180 | Descartados: 60
Bag_8 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
Bag_9 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 183 | Descartados: 57
Bag_10 | Amostras: Sim | Predicados: Não (Todos) | Válidos: 181 | Descartados: 59
PERTURBATION IMPORTANCE PARA PREDICADOS
Valor de perturbação: 0
Métrica: mean_relative_dev
Total de folds: 10


[Bag_1] Processando 183 predicados...
  Predicado: b

/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide




Processando LRC do grafo...


/home/jvribeiro/.local/lib/python3.10/site-packages/networkx/algorithms/centrality/reaching.py:193: RuntimeWarning:

divide by zero encountered in scalar divide



,Predicate_pert_Seed_0,Predicate_pert_Seed_1,Predicate_pert_Seed_2,Predicate_pert_Seed_3
0,Ca ka <= -0.47,Ca ka <= -0.47,Ca ka <= -0.47,Fe ka <= -0.44
1,Ca ka <= -0.22,Mn <= -0.23,Ca ka > -0.22,Si > 0.16
2,Ca ka > -0.22,Si > 0.16,Ca ka <= -0.22,Mn > -0.23
3,Fe ka <= -0.44,Ca ka <= -0.22,Ca ka > -0.72,Ca ka <= -0.47
4,Ca ka > -0.47,Mn > -0.14,Fe ka <= 1.45,Ca ka > -0.22
...,...,...,...,...
190,NaN,Ti ka > 0.50,S <= -0.09,Fe ka + Ti ka <= -0.13
191,NaN,Class_A,background6 > 0.14,Class_A
192,NaN,Class_B,Ti kb > 0.20,Class_B
193,NaN,NaN,Class_A,NaN


In [11]:
from collections import defaultdict

# Coletar posições de cada predicado em cada seed do lrc_pert_all_seeds_df
positions_dict_lrc = defaultdict(list) # o defaultdict cria listas vazias automaticamente

# Iterar sobre cada coluna do dataframe lrc_pert_all_seeds_df
for col in lrc_pert_all_seeds_df.columns:
    # Pegar os predicados da coluna (não-nulos)
    predicates_in_seed = lrc_pert_all_seeds_df[col].dropna().tolist()
    
    # Para cada predicado, guardar sua posição (1-based)
    for position, predicate in enumerate(predicates_in_seed, start=1):
        positions_dict_lrc[predicate].append(position)

# Calcular média e número de aparições
results_lrc = []
for predicate, positions in positions_dict_lrc.items():
    zone_row = predicates_quantiles[0].loc[predicates_quantiles[0]['rule'] == predicate, 'zone']
    zone_value = zone_row.values[0] if not zone_row.empty else None
    results_lrc.append({
        'Predicate': predicate,
        'Mean_Position': np.mean(positions),
        'Appearances': len(positions),
        'Zone': zone_value
    })

# Ordenar: menor posição média primeiro, mais aparições em caso de empate
ranking_lrc_pert_df = pd.DataFrame(results_lrc).sort_values(
    by=['Mean_Position', 'Appearances'], 
    ascending=[True, False]
).reset_index(drop=True)

# Lista final ordenada
lista_ordenada_lrc = ranking_lrc_pert_df['Predicate'].tolist()
ranking_lrc_pert_df

,Predicate,Mean_Position,Appearances,Zone
0,Ca ka <= -0.47,1.75,4,Ca ka
1,Si > 0.16,9.25,4,Si
2,Ca ka > -0.22,10.25,4,Ca ka
3,Al <= 0.22,13.25,4,Al
4,Mn <= -0.23,15.00,4,Mn
...,...,...,...,...
192,P <= -0.16,183.25,4,P
193,Fe ka + Ti ka <= -0.13,186.00,3,Fe ka + Ti ka
194,background7 > 0.17,189.00,1,background7
195,Class_A,191.75,4,None


In [12]:
ranking_pert_lrc_unique_df = ranking_lrc_pert_df.drop_duplicates(subset=['Zone'], keep='first').reset_index(drop=True)
ranking_pert_lrc_unique_df['Zone']

0             Ca ka
1                Si
2                Al
3                Mn
4       background3
5             Fe ka
6       background9
7       background4
8      background12
9       background6
10     background10
11                K
12      background1
13            Ca kb
14      background2
15            Fe kb
16     background11
17        Rh L + Ar
18      background5
19      background7
20                P
21            Ti ka
22            Ti kb
23      background8
24           sum Fe
25               Cu
26               Cr
27    Fe ka + Ti ka
28     background13
29                S
30             None
Name: Zone, dtype: object

In [13]:
with pd.ExcelWriter('Perturbation_method.xlsx') as writer:
    # Primeiro: salvar os rankings médios
    ranking_lrc_pert_df.to_excel(writer, sheet_name='Mean_LRC', index=False)
    ranking_pert_lrc_unique_df.to_excel(writer, sheet_name='Mean_LRC_Unique', index=False)
    
    # Segundo: salvar os LRCs por seed
    for seed in random_seeds:
        lrc_pert_by_seed[seed].to_excel(writer, sheet_name=f'LRC_seed_{seed}', index=False)
    
    # Terceiro: salvar os resultados de perturbação por bag e seed
    for seed in random_seeds:
        for bag_name, df in all_results_pert[seed]['pert_results_dict'].items():
            # Criar nome único: seed_0_Bag_1, seed_1_Bag_1, etc.
            sheet_name = f'seed_{seed}_{bag_name}'[:31]  # Excel limita a 31 caracteres
            df.to_excel(writer, sheet_name=sheet_name, index=False)

# **Permutation**

In [14]:
ranking_perm_lrc_unique_df = pd.read_excel('Permutation_method.xlsx', sheet_name='Mean_LRC_Unique')
ranking_perm_lrc_unique_df['Zone']

0             Ca ka
1                Mn
2                Si
3             Fe ka
4             Ti ka
5             Fe kb
6                 K
7             Ca kb
8             Ti kb
9                Al
10     background11
11                P
12               Cu
13     background12
14     background10
15     background13
16    Fe ka + Ti ka
17        Rh L + Ar
18                S
19               Cr
20           sum Fe
21      background8
22      background7
23      background4
24      background2
25      background1
26      background5
27      background6
28      background3
29      background9
30              NaN
Name: Zone, dtype: object

In [15]:
shap_unique_df = pd.read_csv('shap_soil.csv', sep=';') # loading previously saved shap_unique_df

In [16]:
import numpy as np

max_len = max(
    len(vip_scores_unique_df['Zone']),
    len(reg_vet_unique_df['Zone']),
    len(shap_unique_df['Zone']),
    len(ranking_pert_lrc_unique_df['Zone']),
    len(ranking_perm_lrc_unique_df['Zone']),
    len(ranking_cov_lrc_unique_df['Zone'])
)

def pad_list(lst, length):
    return list(lst) + [None] * (length - len(lst))

features_importance = pd.DataFrame({
    'Vip': pad_list(vip_scores_unique_df['Zone'], max_len),
    'Reg_coef': pad_list(reg_vet_unique_df['Zone'], max_len),
    'Shap': pad_list(shap_unique_df['Zone'], max_len),
    'LRC_cov' : pad_list(ranking_cov_lrc_unique_df['Zone'], max_len),
    'LRC_pert' : pad_list(ranking_pert_lrc_unique_df['Zone'], max_len),
    'LRC_perm' : pad_list(ranking_perm_lrc_unique_df['Zone'], max_len)
})

features_importance.to_csv('feature_importance.csv', index=False, sep=';')
features_importance

,Vip,Reg_coef,Shap,LRC_cov,LRC_pert,LRC_perm
0,Ca ka,Si,Ca ka,Ca ka,Ca ka,Ca ka
1,Fe ka,Mn,Mn,Fe ka,Si,Mn
2,Mn,Ca ka,Si,Ca kb,Al,Si
3,Si,Ti ka,Fe ka,Al,Mn,Fe ka
4,Fe kb,P,Ti ka,K,background3,Ti ka
5,Ti ka,Fe ka,Al,Si,Fe ka,Fe kb
6,Ca kb,Al,Fe kb,Mn,background9,K
7,Al,Ca kb,P,Ti ka,background4,Ca kb
8,K,background11,K,Ti kb,background12,Ti kb
9,sum Fe,background8,background12,Fe kb,background6,Al


In [17]:
# RBO (Rank-Biased Overlap) para comparar rankings
import rbo
rbo_results = {}
reference_list = [x for x in features_importance['Vip'].tolist() if x is not None]
methods = ['Reg_coef', 'Shap', 'LRC_cov', 'LRC_pert', 'LRC_perm']
for method in methods:
    compare_list = [x for x in features_importance[method].tolist() if x is not None]
    # Truncate both lists to the same length (minimum of both)
    min_len = min(len(reference_list), len(compare_list))
    ref_trunc = reference_list[:min_len]
    cmp_trunc = compare_list[:min_len]
    score = rbo.RankingSimilarity(ref_trunc, cmp_trunc).rbo(p=0.7, k=10)
    rbo_results[method] = score
rbo_results = pd.DataFrame(list(rbo_results.items()), columns=['Method','RBO_Score'])
rbo_results.insert(0, 'Reference', 'Vip')
rbo_results.sort_values(by='RBO_Score', ascending=False, inplace=True)
rbo_results.to_csv('rbo_rank.csv', index=False, sep=';')
rbo_results

,Reference,Method,RBO_Score
4,Vip,LRC_perm,0.792084
2,Vip,LRC_cov,0.786569
1,Vip,Shap,0.782470
3,Vip,LRC_pert,0.659278
0,Vip,Reg_coef,0.329164
